In [19]:
import pandas as pd
import numpy as np
import re
import pickle
import matplotlib.pyplot as plt

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.utils import class_weight
from tensorflow.keras.layers import SpatialDropout1D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

In [20]:
data = pd.read_csv("Mental_Health_Condition_Classification.csv", sep=",", engine="python")
print(data.head())
print("Labels:", data['status'].unique())

                                                text                status
0  "My mind is a never-ending cycle of worry, and...               anxiety
1  Despite the sun shining and birds singing outs...               bipolar
2  I'm drowning in responsibilities, each one dem...                stress
3  "My emotions shift like the wind, leaving me u...  personality disorder
4  I'm trapped in a whirlwind of thoughts, unable...               anxiety
Labels: ['anxiety' 'bipolar' 'stress' 'personality disorder' 'normal' 'depression'
 'suicidal']


In [21]:
data['status'].value_counts()

status
anxiety                 17620
normal                  16068
depression              15901
stress                  15230
personality disorder    13915
bipolar                 13708
suicidal                11046
Name: count, dtype: int64

In [22]:
def map_labels(label):
    if label == "anxiety":
        return "anxiety"
    elif label == "depression":
        return "depression"
    elif label == "stress":
        return "stress"
    else:
        return "REMOVE"

data['status_mapped'] = data['status'].apply(map_labels)

data = data[data['status_mapped'] != "REMOVE"]

data['status'].value_counts()

status
anxiety       17620
depression    15901
stress        15230
Name: count, dtype: int64

In [23]:
le = LabelEncoder()
data['label'] = le.fit_transform(data['status_mapped'])
print("Classes mapping:", dict(zip(le.classes_, le.transform(le.classes_))))

Classes mapping: {'anxiety': np.int64(0), 'depression': np.int64(1), 'stress': np.int64(2)}


In [24]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

data['clean_text'] = data['text'].apply(clean_text)

In [25]:
MAX_WORDS = 15000
MAX_LEN = 150

In [26]:
X_train_text, X_val_text, y_train, y_val = train_test_split(
    data['clean_text'], data['label'], test_size=0.2, random_state=42
)

tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train_text)

X_train = pad_sequences(tokenizer.texts_to_sequences(X_train_text), maxlen=MAX_LEN, padding='post', truncating='post')
X_val   = pad_sequences(tokenizer.texts_to_sequences(X_val_text),   maxlen=MAX_LEN, padding='post', truncating='post')

y_train_cat = to_categorical(y_train, num_classes=3)
y_val_cat   = to_categorical(y_val,   num_classes=3)

class_weights = class_weight.compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weights_dict = dict(enumerate(class_weights))

with open('tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)
with open('label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)

print(f'Train: {len(X_train):,}  Val: {len(X_val):,}')
print('Class weights:', class_weights_dict)


Train: 39,000  Val: 9,751
Class weights: {0: np.float64(0.9198330149295973), 1: np.float64(1.0262887818741613), 2: np.float64(1.0655737704918034)}


In [27]:
model = Sequential()

model.add(Embedding(input_dim=MAX_WORDS, output_dim=128, input_length=MAX_LEN))
model.add(SpatialDropout1D(0.3))
model.add(Bidirectional(LSTM(64)))
model.add(Dropout(0.5))
model.add(Dense(32, activation='relu'))
model.add(Dropout(0.3))

model.add(Dense(3, activation='softmax'))

optimizer = Adam(learning_rate=0.0005)
model.compile( loss='categorical_crossentropy', optimizer=optimizer, metrics=['accuracy'])

model.build(input_shape=(None, MAX_LEN))
model.summary()

D:\anac\Lib\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding_6 (Embedding)              │ (None, 150, 128)            │       1,920,000 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ spatial_dropout1d_6                  │ (None, 150, 128)            │               0 │
│ (SpatialDropout1D)                   │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ bidirectional_6 (Bidirectional)      │ (None, 128)                 │          98,816 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_12 (Dropout)                 │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_12 (Dense)                     │ (None, 32)                  │           4,128 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_13 (Dropout)                 │ (None, 32)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_13 (Dense)                     │ (None, 3)                   │              99 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 2,023,043 (7.72 MB)

 Trainable params: 2,023,043 (7.72 MB)

 Non-trainable params: 0 (0.00 B)

In [28]:
from sklearn.metrics import classification_report

early_stop = EarlyStopping(monitor='val_accuracy', patience=4, restore_best_weights=True, verbose=1)
reduce_lr  = ReduceLROnPlateau(monitor='val_accuracy', factor=0.5, patience=2, min_lr=0.00001, verbose=1)

print('Starting Training...')
history = model.fit(
    X_train, y_train_cat,
    epochs=30,
    batch_size=64,
    validation_data=(X_val, y_val_cat),
    callbacks=[early_stop, reduce_lr],
    class_weight=class_weights_dict
)

_, train_acc = model.evaluate(X_train, y_train_cat, verbose=0)
_, val_acc   = model.evaluate(X_val,   y_val_cat,   verbose=0)
print(f'\n----------------------------------------')
print(f'Final Training Accuracy:   {train_acc*100:.2f}%')
print(f'Final Validation Accuracy: {val_acc*100:.2f}%')
print(f'----------------------------------------')

# Classification report
y_pred = np.argmax(model.predict(X_val, verbose=0), axis=1)
print('\n', classification_report(y_val, y_pred, target_names=le.classes_))


Starting Training...
Epoch 1/30
610/610 ━━━━━━━━━━━━━━━━━━━━ 104s 148ms/step - accuracy: 0.8851 - loss: 0.3065 - val_accuracy: 0.9466 - val_loss: 0.1456 - learning_rate: 5.0000e-04
Epoch 2/30
610/610 ━━━━━━━━━━━━━━━━━━━━ 81s 132ms/step - accuracy: 0.9584 - loss: 0.1288 - val_accuracy: 0.9545 - val_loss: 0.1279 - learning_rate: 5.0000e-04
Epoch 3/30
610/610 ━━━━━━━━━━━━━━━━━━━━ 77s 126ms/step - accuracy: 0.9712 - loss: 0.0908 - val_accuracy: 0.9541 - val_loss: 0.1300 - learning_rate: 5.0000e-04
Epoch 4/30
610/610 ━━━━━━━━━━━━━━━━━━━━ 0s 117ms/step - accuracy: 0.9796 - loss: 0.0685  
Epoch 4: ReduceLROnPlateau reducing learning rate to 0.0002500000118743628.
610/610 ━━━━━━━━━━━━━━━━━━━━ 78s 128ms/step - accuracy: 0.9784 - loss: 0.0712 - val_accuracy: 0.9522 - val_loss: 0.1410 - learning_rate: 5.0000e-04
Epoch 5/30
610/610 ━━━━━━━━━━━━━━━━━━━━ 89s 145ms/step - accuracy: 0.9863 - loss: 0.0476 - val_accuracy: 0.9547 - val_loss: 0.1596 - learning_rate: 2.5000e-04
Epoch 6/30
610/610 ━━━━━━━━━

In [29]:
import numpy as np
from tensorflow.keras.preprocessing.sequence import pad_sequences

def test_my_bot(sentence):
    print("\n" + "="*40)
    print(f"🗣️ المريض بيقول: '{sentence}'")
    print("="*40)

    clean_s = clean_text(sentence)

    seq = tokenizer.texts_to_sequences([clean_s])
    padded = pad_sequences(seq, maxlen=MAX_LEN, padding='post', truncating='post')

    pred_probs = model.predict(padded, verbose=0)[0]
    pred_class_index = np.argmax(pred_probs)

    predicted_label = le.inverse_transform([pred_class_index])[0]
    confidence = pred_probs[pred_class_index] * 100

    print(f"🤖 تشخيص الموديل: {predicted_label} (بنسبة ثقة {confidence:.2f}%)")
    print("\n📊 تفاصيل التفكير:")
    for i, class_name in enumerate(le.classes_):
        print(f"  - {class_name}: {pred_probs[i]*100:.2f}%")



test_my_bot("My heart is beating so fast, I can't breathe, and I feel like I'm going to die right now.")

test_my_bot("I just feel so empty. Nothing makes me happy anymore, I just want to stay in bed all day and sleep forever.")

test_my_bot("I have 3 exams tomorrow, my laptop just broke, and I'm totally overwhelmed with all this work!")

test_my_bot("I can't stop thinking about what might go wrong, my hands are shaking and I feel like I can't catch my breath.")

test_my_bot("Everything feels incredibly heavy today. I didn't even get out of bed, there is just no point in trying anymore.")

test_my_bot("My boss is demanding the report by 5 PM, my phone won't stop ringing, and I haven't even had time to eat lunch!")

test_my_bot("I am exhausted from working so many hours, but when I try to sleep, my brain won't shut off and I panic about tomorrow.")



🗣️ المريض بيقول: 'My heart is beating so fast, I can't breathe, and I feel like I'm going to die right now.'
🤖 تشخيص الموديل: anxiety (بنسبة ثقة 95.63%)

📊 تفاصيل التفكير:
  - anxiety: 95.63%
  - depression: 2.43%
  - stress: 1.94%

🗣️ المريض بيقول: 'I just feel so empty. Nothing makes me happy anymore, I just want to stay in bed all day and sleep forever.'
🤖 تشخيص الموديل: depression (بنسبة ثقة 99.99%)

📊 تفاصيل التفكير:
  - anxiety: 0.00%
  - depression: 99.99%
  - stress: 0.00%

🗣️ المريض بيقول: 'I have 3 exams tomorrow, my laptop just broke, and I'm totally overwhelmed with all this work!'
🤖 تشخيص الموديل: stress (بنسبة ثقة 79.60%)

📊 تفاصيل التفكير:
  - anxiety: 6.71%
  - depression: 13.69%
  - stress: 79.60%

🗣️ المريض بيقول: 'I can't stop thinking about what might go wrong, my hands are shaking and I feel like I can't catch my breath.'
🤖 تشخيص الموديل: anxiety (بنسبة ثقة 86.65%)

📊 تفاصيل التفكير:
  - anxiety: 86.65%
  - depression: 1.48%
  - stress: 11.87%

🗣️ المريض بيقول: 'E

In [30]:
model.save('mental_health_nlp_model.h5')
print('Saved!')

Saved!
